# Test API Best Express Việt Nam (môi trường TEST)

Chạy lần lượt từng ô. Chỉ dùng **môi trường test**. Tài khoản nhập qua biến môi trường `BEST_USER` / `BEST_PASS` hoặc hỏi khi chạy. Đơn tạo ra là đơn test và được hủy ở bước cuối.

| Bước | Endpoint |
|---|---|
| 1 | `User/Login` lấy token |
| 2 | `Service/EstimateFee` cước tạm tính |
| 3 | `Order/Add` tạo đơn |
| 4 | `Order/Query` tra hành trình (host test riêng) |
| 5 | `Print/PrintWayBill` in vận đơn |
| 6 | `Order/Update` sửa đơn |
| 7 | `Order/Cancel` hủy đơn |

In [ ]:
import os, json, time, getpass, requests

BASE_TEST  = "http://sgp-seaedi-test.800best.com/VietNamV3/v3/api/process/sears"
# Tài liệu ghi riêng một host test cho TraceQuery
QUERY_TEST = "http://sea-edi-hxtest.800best.com/VietNamV3/v3/api/process/sears"

# Không ghi mật khẩu vào file: đọc từ biến môi trường BEST_USER / BEST_PASS, hoặc hỏi khi chạy.
HAS_TOKEN = bool(os.environ.get("BEST_TOKEN"))
ACCOUNT = {
    "Username": os.environ.get("BEST_USER") or ("" if HAS_TOKEN else input("Best username: ")),
    "Password": os.environ.get("BEST_PASS") or ("" if HAS_TOKEN else getpass.getpass("Best password: ")),
}

def call(url, body, token=None):
    headers = {"Content-Type": "application/json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    r = requests.post(url, headers=headers, data=json.dumps(body), timeout=30)
    print("HTTP", r.status_code, "|", url.split("/sears/")[1])
    try:
        data = r.json()
    except ValueError:
        data = {"raw": r.text[:500]}
    print(json.dumps(data, ensure_ascii=False, indent=2)[:3000])
    return data

## 1. Đăng nhập lấy token (hiệu lực 30 ngày)

In [ ]:
TOKEN = os.environ.get("BEST_TOKEN")   # có sẵn token thì bỏ qua đăng nhập
if not TOKEN:
    res = call(f"{BASE_TEST}/User/Login", ACCOUNT)
    TOKEN = res.get("token")
assert TOKEN, "Không lấy được token, xem phản hồi ở trên"
print("Token:", TOKEN[:20] + "...")

## 2. Cước tạm tính
Tên địa danh phải đúng như hệ thống Best (địa chỉ cũ 3 cấp, hoặc mới 2 cấp: `DestDistrict` để trống).

In [ ]:
fee_body = {
    "UserName": ACCOUNT["Username"],
    "ServiceId": 12491,                # Giao hàng tiết kiệm (dịch vụ duy nhất còn hỗ trợ)
    "SourceCity": "Hà Nội", "SourceDistrict": "Quận Thanh Xuân", "SourceWard": "Phường Thượng Đình",
    "DestCity": "Hồ Chí Minh", "DestDistrict": "Tân Bình", "DestWard": "Phường 13",
    "Cod": 180000, "ProductPrice": 180000,
    "Weight": 600, "Length": 140, "Height": 150, "Width": 200,
}
fee = call(f"{BASE_TEST}/Service/EstimateFee", fee_body, TOKEN)

## 3. Tạo đơn hàng test

In [ ]:
ORDER_CODE = f"TEST{int(time.time())}"   # mã đơn của mình, phải khác nhau mỗi lần
order_body = {
    "Code": ORDER_CODE,
    "ProductName": "Đơn test - Rèm cửa",
    "CollectAmount": 180000, "ProductPrice": 180000,
    "JourneyType": 1, "ServiceId": 12491, "PickupType": 2,
    "Weight": 600, "Height": 150, "Length": 140, "Width": 200,
    "SourceCity": "Hà Nội", "SourceDistrict": "Quận Thanh Xuân", "SourceWard": "Phường Thượng Đình",
    "SourceAddress": "Đường Nguyễn Trãi, Phường Thượng Đình, Quận Thanh Xuân, Hà Nội",
    "SourceName": "Phùng Ngọc Hà", "SourcePhoneNumber": "1900 1031",
    "DestCity": "Hồ Chí Minh", "DestDistrict": "Tân Bình", "DestWard": "Phường 13",
    "DestAddress": "Cộng Hòa, Quận Tân Bình, Phường 13, Quận Tân Bình, Hồ Chí Minh",
    "DestName": "Nguyễn Quốc Thuận", "DestPhoneNumber": "1900 1031",
    "ReturnCity": "Hà Nội", "ReturnDistrict": "Quận Thanh Xuân", "ReturnWard": "Phường Thượng Đình",
    "ReturnAddress": "Đường Nguyễn Trãi, Phường Thượng Đình, Quận Thanh Xuân, Hà Nội",
    "ReturnName": "Phùng Ngọc Hà", "ReturnPhoneNumber": "1900 1031",
    "Note": "Đây là đơn hàng test. Không cần thực hiện bất cứ hành động nào đối với đơn hàng này",
    "ShipmentPayer": 0,
}
created = call(f"{BASE_TEST}/Order/Add", order_body, TOKEN)
BEST_CODE = created.get("Code")   # mã tra cứu hành trình của Best
print("Result:", created.get("Result"), "(1 = thành công, 2 = thất bại) | Best code:", BEST_CODE)

## 4. Tra cứu hành trình
Tối đa 10 mã mỗi lần. Đơn mới tạo có thể chưa có hành trình.

In [ ]:
trace = call(f"{QUERY_TEST}/Order/Query",
             {"LangType": "en-US", "Codes": [c for c in (BEST_CODE, ORDER_CODE) if c]}, TOKEN)

## 5. In vận đơn (trả về link PDF)

In [ ]:
pdf = call(f"{BASE_TEST}/Print/PrintWayBill", {"codes": [BEST_CODE]}, TOKEN)
print("PDF:", pdf.get("Url"))

## 6. Cập nhật đơn (chỉ được khi chưa lấy hàng)

In [ ]:
upd = call(f"{BASE_TEST}/Order/Update", {
    "Code": BEST_CODE or ORDER_CODE,
    "ProductName": "Đơn test - Rèm cửa (đã sửa)",
    "CollectAmount": 150000, "ProductPrice": 180000,
    "ServiceId": 12491, "Weight": 700, "Width": 200, "Height": 150, "Length": 140,
}, TOKEN)

## 7. Hủy đơn (dọn đơn test)
Chỉ hủy được khi shipper chưa lấy hàng thành công.

In [ ]:
cancel = call(f"{BASE_TEST}/Order/Cancel", {"Code": BEST_CODE or ORDER_CODE}, TOKEN)